# Kappa-scale calibration on the Colab T4

Calibrates the single free scale parameter of a heterogeneous
capital-productivity Krusell-Smith economy: the cross-sectional *shape* of
`kappa` is hardcoded from Xavier (2021)'s return-on-wealth-by-percentile
curve (already cited in the paper), and `k_multiplier` is adaptively searched
(a (1+1) evolution strategy, not a flat grid -- see README.md "Search
strategy") to score how closely each trained economy's own emergent
steady-state return distribution reproduces that same curve. Full design
writeup: `runs/ks-heterogeneous-returns/README.md`.

Default config: `n_agents=1000`, `num_envs=8` (deliberately less than
`ks_n200.yaml`'s own 32 -- puts the batch width `num_envs*n_agents=8,000`
inside the scaling mesh's measured plateau instead of past its knee; see
README.md "Batch width"), 1 initial guess + `es_iters=12` adaptive
evaluations. Each evaluation is a full training run on the same protocol as
`configs/exp/ks_n200.yaml`/`ks_n2000.yaml` otherwise -- **time the first
evaluation before assuming the rest fit in one Colab session**; n=1000 is
noticeably more expensive than the ~3 min/cell the n=200 correctness run
reports. Every evaluation prints the actual envs/agents/memory footprint
being used (no more guessing from how slow it feels), and results.csv is
checkpointed after every evaluation so a disconnect partway through loses
only the run in progress.


In [ ]:
# Setup: clone or update the repo, install (idempotent -- safe to re-run).
%cd /content
![ -d jax-marl-bc ] || git clone https://github.com/danmonuni/jax-marl-bc.git
%cd jax-marl-bc
!git pull
!pip install -q -r requirements.txt && pip install -q -e . --no-deps


In [ ]:
# Sanity: a GPU runtime is attached (Runtime > Change runtime type > T4 GPU).
!nvidia-smi -L


In [ ]:
# Mount Drive BEFORE the run so the result is saved as soon as it finishes
# (a Colab disconnect then loses at most the run in progress, never a
# finished one).
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

def save_results(name='ks-heterogeneous-returns'):
    """Sync runs/<name>/results -> Drive (exact path, idempotent re-sync)."""
    src = f'runs/{name}/results'
    dst = f'/content/drive/MyDrive/jax-marl-bc-runs/{name}/results'
    assert os.path.exists(src), f"{src} missing - did the calibration run finish?"
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"saved {src} -> {dst}")


## Calibration search

Runs `runs/ks-heterogeneous-returns/config.yaml` as-is: `mode=es`,
`n_agents=1000`, `device=gpu`, starting from the naive
`1/mean(base_vector) ~ 0.26` guess. Pass dotlist overrides after the script
path to change any of these -- e.g. fewer iterations for a first timing
check, or the flat-grid fallback:
`!python runs/ks-heterogeneous-returns/calibrate_kappa_scale.py es_iters=2`
`!python runs/ks-heterogeneous-returns/calibrate_kappa_scale.py mode=grid "k_grid=[0.2,0.3]"`


In [ ]:
!python runs/ks-heterogeneous-returns/calibrate_kappa_scale.py


In [ ]:
save_results('ks-heterogeneous-returns')


## Results


In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv('runs/ks-heterogeneous-returns/results/results.csv')
display(df[['k_multiplier', 'score_rms_ratio_minus_1', 'capital_gini', 'top_0.1_share']])

fig_path = 'runs/ks-heterogeneous-returns/results/calibration_fit.png'
display(Image(fig_path))
